In [ ]:
# 01_eda.ipynb -- Era 3 (Nov 2024-present) SCADA EDA: violation + ramp-shock rate by
# hour, season, gen-mix, and corridor stress
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip
# !pip install lightgbm -q

import pandas as pd
import matplotlib.pyplot as plt

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)

df = f.drop_bad_days(scada)  # drops 2024-11-20, 2025-04-01 (1 slot), 2025-10-02 (63 slots)
df = f.add_datetime(df)
df = f.add_violation_label(df)
df = f.add_ramp_label(df)
df = f.add_time_features(df)

print("rows after dropping bad days:", len(df), "of", len(scada))
print("overall violation rate:", round(df["violation"].mean(), 4))
print("overall ramp rate:", round(df["ramp"].mean(), 4))

# --- Rate by hour of day ---
by_hour = df.groupby("hour")[["violation", "ramp"]].mean()
import matplotlib as mpl
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

norm_v = mpl.colors.Normalize(vmin=by_hour["violation"].min(), vmax=by_hour["violation"].max())
axes[0].plot(by_hour.index, by_hour["violation"], color="#4a3aa7", alpha=0.25, linewidth=1)
axes[0].scatter(by_hour.index, by_hour["violation"], c=by_hour["violation"], cmap="RdPu", s=40, zorder=3)
axes[0].set_title("Frequency-violation rate by hour")

norm_r = mpl.colors.Normalize(vmin=by_hour["ramp"].min(), vmax=by_hour["ramp"].max())
axes[1].plot(by_hour.index, by_hour["ramp"], color="#4a3aa7", alpha=0.25, linewidth=1)
axes[1].scatter(by_hour.index, by_hour["ramp"], c=by_hour["ramp"], cmap="cool", s=40, zorder=3)
axes[1].set_title("Ramp-shock rate by hour")
plt.tight_layout()
plt.savefig("era3_rate_by_hour.png")
print(by_hour.round(4))

# --- Rate by month (seasonality) ---
by_month = df.groupby("month")[["violation", "ramp"]].mean()
print("\n", by_month.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
norm_v = mpl.colors.Normalize(vmin=by_month["violation"].min(), vmax=by_month["violation"].max())
axes[0].bar(by_month.index.astype(str), by_month["violation"], color=mpl.colormaps["RdPu"](norm_v(by_month["violation"].values)))
axes[0].set_title("Violation rate by month")
axes[0].set_xlabel("month")

norm_r = mpl.colors.Normalize(vmin=by_month["ramp"].min(), vmax=by_month["ramp"].max())
axes[1].bar(by_month.index.astype(str), by_month["ramp"], color=mpl.colormaps["cool"](norm_r(by_month["ramp"].values)))
axes[1].set_title("Ramp-shock rate by month")
axes[1].set_xlabel("month")

plt.tight_layout()
plt.savefig("era3_rate_by_month.png")
plt.show()

# --- Solar-hour vs non-solar-hour ---
print("\nviolation rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["violation"].mean().round(4))
print("ramp rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["ramp"].mean().round(4))

solar_viol = df.groupby("is_solar_hr")["violation"].mean()
solar_ramp = df.groupby("is_solar_hr")["ramp"].mean()
fig, ax = plt.subplots(figsize=(6, 4.5))
x = np.arange(2)
width = 0.35
ax.bar(x - width / 2, [solar_viol[0], solar_viol[1]], width, label="violation rate", color="#e87ba4")
ax.bar(x + width / 2, [solar_ramp[0], solar_ramp[1]], width, label="ramp rate", color="#2a78d6")
ax.set_xticks(x, ["non-solar hour", "solar hour"])
ax.set_ylabel("event rate")
ax.set_title("Event rate: solar vs non-solar hour")
ax.legend()
plt.tight_layout()
plt.savefig("era3_solar_hour_rates.png")
plt.show()

# --- Weekday vs weekend ---
print("\nviolation rate, weekday(0) vs weekend(1):")
print(df.groupby("is_weekend")["violation"].mean().round(4))

# --- Generation mix: RES share vs event rate ---
df["res_bin"] = pd.qcut(df["share_res_pct"], 5, duplicates="drop")
print("\nviolation/ramp rate by RES-share quintile (low -> high):")
print(df.groupby("res_bin", observed=True)[["violation", "ramp"]].mean().round(4))

res_pooled = df.groupby("res_bin", observed=True)[["violation", "ramp"]].mean()
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(res_pooled))
norm = mpl.colors.Normalize(vmin=res_pooled["violation"].min(), vmax=res_pooled["violation"].max())
ax.bar(x, res_pooled["violation"], color=mpl.colormaps["RdPu"](norm(res_pooled["violation"].values)))
ax.set_xticks(x, ["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"])
ax.set_xlabel("RES-share quintile")
ax.set_ylabel("violation rate")
ax.set_title("Violation rate rises with RES share (pooled, SCADA resolution)")
plt.tight_layout()
plt.savefig("era3_res_share_violation.png")
plt.show()

# --- Corridor congestion: sum|ir_net| vs event rate ---
ir_abs_sum = df[f.CORRIDOR_COLS[:7]].abs().sum(axis=1)  # the 7 ir_* corridor cols
df["ir_bin"] = pd.qcut(ir_abs_sum, 5, duplicates="drop")
print("\nviolation/ramp rate by corridor-flow quintile (low -> high):")
print(df.groupby("ir_bin", observed=True)[["violation", "ramp"]].mean().round(4))

# --- Findings (verified 2026-07-11 against live study2_scada.csv, 56,892 usable slots
#     after dropping the 3 corrupted-file days) ---
#
# Overall base rates: violation 0.89%, ramp-shock 6.1% -- both rare-event but workable
# class balances (matches the 0.88% violation rate measured earlier in the roadmap).
#
# STRONG time-of-day pattern, and it lines up with solar ramp physics, not noise:
#   - Violations cluster 07:00-14:00, peaking at 13:00 (4.0%) and 08:00-09:00 (~3%) --
#     the mid-morning-to-early-afternoon window where solar output is both large and
#     fast-changing (cloud transients, ramp-up/plateau).
#   - Ramp-shocks cluster in two bands: 05:00-09:00 (sunrise ramp-up, peaking 36% at
#     06:00) and 17:00-20:00 (sunset ramp-down, up to 8.6%) -- the two times of day solar
#     generation changes fastest. This is a clean, physically-explainable signal, not an
#     artifact -- and it's the single strongest predictor a lead-time model should exploit
#     (confirmed in 03/04's feature importance: "hour" is the top feature for ramp-shock).
#   - Solar-hour violation rate (1.53%) is ~6x the non-solar-hour rate (0.26%).
#
# RES share vs violation rate is MONOTONIC and increasing: 0.33% in the lowest RES-share
# quintile up to 1.92% in the highest -- direct SCADA-resolution evidence for the
# project's central "rising RES share stresses the grid" thesis (Era 1 found a similar,
# weaker signal at daily/monthly resolution; this is the live, granular version of it).
#
# RES share vs ramp-shock rate goes the OTHER way in this simple quintile binning (8.3%
# in lowest quintile down to 2.8% in highest) -- flagged as a genuine, unresolved,
# counter-intuitive finding, not smoothed over: RES share is itself strongly seasonal
# (higher in summer), and month is independently a strong driver of ramp rate (Jan/Dec
# ~13-15%, Jul ~0.7%), so this crude binning is very likely confounded by season rather
# than showing a true RES effect. A cleaner month-controlled analysis is future work, not
# resolved here.
#
# Corridor-flow (ir_abs_sum) quintiles show no clean monotonic relationship with either
# event rate in this simple binning -- consistent with Era 2's daily-resolution finding
# that corridor flow's relationship to stress is real but not simply "more flow = more
# events" (it's corridors correcting stress, not causing it). The lead-time classifiers
# in 03/04 use the raw per-corridor columns rather than this aggregate, which is
# expected to capture more of that relationship than a single summed quintile can.


In [ ]:
# --- Appendix: hour x day-of-week heatmap, and resolving the RES-share/ramp finding
#     (added 2026-07-11) ---
# The single-axis breakdowns above (hour alone, month alone) can hide interaction
# effects. This cross-tabs hour against day-of-week, and separately resolves the
# "ramp rate falls with RES-share quintile" finding flagged above as likely
# season-confounded -- by actually controlling for season instead of leaving it open.

import numpy as np
import matplotlib.pyplot as plt

print("=== violation rate: hour (columns) x day-of-week (rows, 0=Mon..6=Sun) ===")
viol_heatmap = df.pivot_table(index="dow", columns="hour", values="violation", aggfunc="mean")
print(viol_heatmap.round(3).to_string())

print("\n=== ramp rate: hour (columns) x day-of-week (rows, 0=Mon..6=Sun) ===")
ramp_heatmap = df.pivot_table(index="dow", columns="hour", values="ramp", aggfunc="mean")
print(ramp_heatmap.round(3).to_string())

print("\nmarginal violation rate by day-of-week:", df.groupby("dow")["violation"].mean().round(4).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
im0 = axes[0].imshow(viol_heatmap.values, cmap="RdPu", aspect="auto")
axes[0].set_xticks(range(len(viol_heatmap.columns)), viol_heatmap.columns)
axes[0].set_yticks(range(len(viol_heatmap.index)), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[0].set_xlabel("hour")
axes[0].set_title("Violation rate: hour x day-of-week")
fig.colorbar(im0, ax=axes[0], label="violation rate")

im1 = axes[1].imshow(ramp_heatmap.values, cmap="cool", aspect="auto")
axes[1].set_xticks(range(len(ramp_heatmap.columns)), ramp_heatmap.columns)
axes[1].set_yticks(range(len(ramp_heatmap.index)), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].set_xlabel("hour")
axes[1].set_title("Ramp-shock rate: hour x day-of-week")
fig.colorbar(im1, ax=axes[1], label="ramp rate")

plt.tight_layout()
plt.savefig("era3_heatmap_hour_dow.png")
plt.show()
print("marginal ramp rate by day-of-week:", df.groupby("dow")["ramp"].mean().round(4).to_dict())

# --- Season-controlled RES-share vs ramp/violation rate ---
# Instead of a global RES-share quintile (confounded by season -- RES share and ramp
# rate are both independently seasonal), rank RES share WITHIN each month first, then
# quintile that rank. This isolates "is this day unusually high/low RES for its own
# season" from "which month is it."
df["res_rank_in_month"] = df.groupby("month")["share_res_pct"].rank(pct=True)
df["res_bin_within_month"] = pd.cut(df["res_rank_in_month"], 5, labels=False)
within_month = df.groupby("res_bin_within_month", observed=True)[["violation", "ramp"]].mean()
print("\nviolation/ramp rate by RES-share quintile, WITHIN month (season-controlled):")
print(within_month.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(5)
width = 0.35
pooled_ramp = df.groupby("res_bin", observed=True)["ramp"].mean().values
ax.bar(x - width / 2, pooled_ramp, width, label="pooled (uncontrolled)", color="#9085e9")
ax.bar(x + width / 2, within_month["ramp"].values, width, label="within-month (season-controlled)", color="#4a3aa7")
ax.set_xticks(x, ["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"])
ax.set_xlabel("RES-share quintile")
ax.set_ylabel("ramp-shock rate")
ax.set_title("RES-share vs ramp-shock rate: pooled sign reverses once season is controlled for")
ax.legend()
plt.tight_layout()
plt.savefig("era3_res_share_ramp_reversal.png")
plt.show()

# Partial correlation: residualize both RES share and the outcome against month dummies
# (linear regression on month, keep the residuals), then correlate the residuals. This
# is the same idea as the within-month quintile above but as a single number.
month_dummies = pd.get_dummies(df["month"], prefix="m", drop_first=True).astype(float)
X = np.column_stack([np.ones(len(df)), month_dummies.values])

def residualize(y):
    y = y.values.astype(float)
    mask = ~np.isnan(y)
    beta, *_ = np.linalg.lstsq(X[mask], y[mask], rcond=None)
    resid = np.full(len(y), np.nan)
    resid[mask] = y[mask] - X[mask] @ beta
    return resid

res_share_resid = residualize(df["share_res_pct"])
ramp_resid = residualize(df["ramp"])
viol_resid = residualize(df["violation"])
m1 = ~np.isnan(res_share_resid) & ~np.isnan(ramp_resid)
m2 = ~np.isnan(res_share_resid) & ~np.isnan(viol_resid)
print("\npooled corr(share_res_pct, ramp):", round(df[["share_res_pct", "ramp"]].corr().iloc[0, 1], 4))
print("partial corr(share_res_pct, ramp | month):", round(np.corrcoef(res_share_resid[m1], ramp_resid[m1])[0, 1], 4))
print("partial corr(share_res_pct, violation | month):", round(np.corrcoef(res_share_resid[m2], viol_resid[m2])[0, 1], 4))

# --- Findings (verified 2026-07-11) ---
# Heatmap: violations concentrate hardest on Sunday (dow=6) -- the single worst slot is
# Sunday 13:00 at 7.0%, and Sunday's marginal violation rate (1.38%) is the highest of
# any day, well above the weekday range (0.62-0.99%). Ramp-shocks show the opposite
# day-of-week pattern: Sunday's marginal ramp rate (4.40%) is the LOWEST of the week,
# while every weekday sits around 6.4-6.7%. That's a genuinely interesting decoupling,
# not noise: Sunday has fewer large demand swings (lower industrial/commercial load,
# smoother curve) but MORE frequency violations -- consistent with the grid running a
# thinner online generation/reserve margin on low-demand days, making frequency more
# sensitive to whatever smaller disturbances do occur. Worth carrying into feature
# engineering or the paper's discussion, not just filed as a curiosity.
#
# RES-share/ramp confounding, RESOLVED: the earlier pooled finding (ramp rate falls
# from 8.3% to 2.8% across RES-share quintiles) was indeed season-confounded, as
# suspected -- and controlling for it doesn't just weaken the effect, it REVERSES it.
# Within-month (season-controlled) RES-share quintiles show ramp rate RISING from 5.3%
# to 7.0%, and the partial correlation (residualized on month) flips from -0.075
# (pooled) to +0.026 (season-controlled) -- small in magnitude, but the correct sign is
# now consistent with the violation-rate finding (partial corr +0.051) and with the
# project's central "rising RES share stresses the grid" thesis. The pooled/uncontrolled
# number was actively misleading, not just imprecise -- this is a case where the
# season-controlled analysis was necessary to get the right qualitative answer, not just
# a more precise one.
